In [2]:
#This is my random forest algorithm
import yfinance as yf 
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

In [12]:
def is_real_ticker(ticker):
    Stock = yf.Ticker(ticker)
    data = Stock.history(period="1d")
    return not data.empty

Stock = input("Enter a Stock Ticker:")

if not is_real_ticker(Stock):
    print("Invalid Stock Ticker. Please try again.")
    exit()

    
data = yf.download(Stock, start="2010-01-01", end=datetime.now().strftime("%Y-%m-%d"))

if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.droplevel(1)

data['Return'] = data['Close'].pct_change()

data['Target'] = (data["Return"].shift(-1) > 0 ).astype(int)

data['Return_lag1'] = data['Return'].shift(1)
data['Return_lag2'] = data['Return'].shift(2)
data['Return_lag5'] = data['Return'].shift(5)
data['Return_lag10'] = data['Return'].shift(10)

data['MA5'] = data['Close'].rolling(5).mean()
data['MA10'] = data['Close'].rolling(10).mean()

data['Volatility'] = data['Return'].rolling(10).std()

data["Price_MA5"] = (data["Close"] - data["MA5"])/ data["MA5"]

data["Volume_change"] = data["Volume"].pct_change()

data["Overnight_Gap"] = (data['Open'] - data['Close'].shift(1)) / data['Close'].shift(1)

data["Ratio_20MA"] = data['Close'] / data['Close'].rolling(window=20).mean()

data["Dist-10H"] = (data['High'].rolling(window=10).max() - data['Close']) / data['Close']

data["AvgVOL-5"] = (data['Volume_change']- data['Volume_change'].rolling(5).mean()) / data['Volume_change'].rolling(5).mean()

data["AvgVOLT-5"] = (data['Volatility'] - data['Volatility'].rolling(5).mean()) / data['Volatility'].rolling(5).mean()

data = data.dropna()





    


[*********************100%***********************]  1 of 1 completed


In [13]:
X = data[[
    'Return_lag1', 
    'Return_lag2', 
    'Return_lag5', 
    'Return_lag10',
    'MA5', 
    'MA10', 
    'Volatility', 
    'Price_MA5', 
    'Volume_change',
    'Overnight_Gap', 
    'Ratio_20MA', 
    'Dist-10H', 
    'AvgVOL-5', 
    'AvgVOLT-5'
]]

In [15]:
data["Return_NextDay"] = data["Return"].shift(-1)
data["Target"] = (data["Return_NextDay"] > 0).astype(int)
y = data["Target"]

split = int(len(data)*0.8)
X_train = X.iloc[:split]
X_test = X.iloc[split:]
y_train = y.iloc[:split]
y_test = y.iloc[split:]